In [1]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGES = [
    "transformers==4.55.2",
    "peft==0.17.0",
    "trl==0.21.0",
    "bitsandbytes==0.48.0",
    "accelerate==1.10.0",
    "datasets==4.0.0",
    "huggingface-hub<1.0",
]

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    print("설치 완료")
else:
    print("로컬 환경이라 설치를 건너뜁니다.")

설치 완료


In [2]:
import os

# GPU 메모리 단편화를 줄이는 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# 혹시 남아 있을 수 있는 강제 CUDA 버전 지정 해제
os.environ.pop("BNB_CUDA_VERSION", None)

import torch
import transformers
import trl
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("transformers", transformers.__version__, "| trl", trl.__version__)

if transformers.__version__ != "4.55.2" or trl.__version__ != "0.21.0":
    raise RuntimeError("라이브러리 버전이 다릅니다. 메뉴 '런타임 > 세션 다시 시작'을 누른 뒤 이 셀부터 다시 실행하세요.")

transformers 4.55.2 | trl 0.21.0


In [3]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "qwen-glaive-python"

# 원래 4GB GPU에 맞춘 길이입니다. 이보다 긴 샘플은 자르지 않고 아예 뺍니다.
MAX_LENGTH = 256

TRAIN_SIZE = 500
VAL_SIZE = 100

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab 메뉴 '런타임 > 런타임 유형 변경'에서 T4 GPU를 고르세요.")
print(torch.cuda.get_device_name(0))

Tesla T4


In [4]:
raw = load_dataset("glaiveai/glaive-code-assistant", split="train")
raw

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

c9bc9129-eba0-4b10-8292-4ae70fc7fa0d.jso(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/136109 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer'],
    num_rows: 136109
})

In [5]:
python_rows = raw.filter(lambda item: "python" in item["question"].lower())
len(python_rows)

Filter:   0%|          | 0/136109 [00:00<?, ? examples/s]

49219

In [6]:
def messages_for(item):
    message = f"Get answers for questions related to Python. Respond with answers, no explanation\n\n{item['question']}"
    return {
        "prompt": [{"role": "user", "content": message}],
        "completion": [{"role": "assistant", "content": item["answer"]}],
    }

In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def token_length(item):
    m = messages_for(item)
    text = tokenizer.apply_chat_template(m["prompt"] + m["completion"], tokenize=False)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
selected = []
for item in python_rows.shuffle(seed=42):
    if token_length(item) <= MAX_LENGTH:
        selected.append(item)
        if len(selected) == TRAIN_SIZE + VAL_SIZE:
            break

train_set = selected[:TRAIN_SIZE]
val_set = selected[TRAIN_SIZE:]
len(train_set), len(val_set)

(500, 100)

In [9]:
messages_for(val_set[0])

{'prompt': [{'role': 'user',
   'content': 'Get answers for questions related to Python. Respond with answers, no explanation\n\nHow can I create a list in Python that stores the first 10 numbers divisible by 4?'}],
 'completion': [{'role': 'assistant',
   'content': "You can use list comprehension in Python to create such a list. Here's an example code snippet that accomplishes this:\n\n```python\ndivisible_by_four = [i for i in range(1,11) if i % 4 == 0]\n```\n\nIn this code, we use the `range(1, 11)` function to generate numbers from 1 to 10 (inclusive). Then, we use the `%` operator to check if each number is divisible by 4. If the remainder of the division is 0, the number is considered divisible by 4, and it is included in the resulting list `divisible_by_four`."}]}

In [10]:
train_ds = Dataset.from_list([messages_for(item) for item in train_set])
val_ds = Dataset.from_list([messages_for(item) for item in val_set])
train_ds

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 500
})

In [11]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [13]:
print("VRAM: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))

VRAM: 1.50 GB


In [14]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [15]:
train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    completion_only_loss=True,      # 답변 토큰에만 손실을 겁니다
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,                      # Turing 세대(T4, GTX 1650)라 bf16은 못 씁니다
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="epoch",          # 에폭 끝에 검증 손실을 계산합니다
    per_device_eval_batch_size=1,
    save_strategy="epoch",
    report_to=[],
    seed=42,
)

In [16]:
trainer = SFTTrainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.576400,0.561093


TrainOutput(global_step=63, training_loss=0.5576308965682983, metrics={'train_runtime': 350.2019, 'train_samples_per_second': 1.428, 'train_steps_per_second': 0.18, 'total_flos': 858899753395200.0, 'train_loss': 0.5576308965682983})

In [18]:
print("peak VRAM: %.2f GB" % (torch.cuda.max_memory_reserved() / 1024**3))

peak VRAM: 2.58 GB


In [19]:
trainer.save_model(OUTPUT_DIR)

In [20]:
import shutil

# False면 내 PC로 내려받고, True면 Google Drive에 저장합니다.
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/fine-tuning"

if IN_COLAB:
    zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
    if SAVE_TO_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print("Drive에 저장:", shutil.copy(zip_path, DRIVE_DIR))
    else:
        from google.colab import files

        files.download(f"{OUTPUT_DIR}.zip")
else:
    print("로컬 환경이라 내려받기를 건너뜁니다. 결과는", OUTPUT_DIR, "폴더에 있습니다.")


Mounted at /content/drive
Drive에 저장: /content/drive/MyDrive/fine-tuning/qwen-glaive-python.zip


In [21]:
from google.colab import drive
drive.mount("/content/drive")
shutil.unpack_archive("/content/drive/MyDrive/fine-tuning/qwen-glaive-python.zip", "qwen-glaive-python")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
if "trainer" in globals():
    # 방금 학습한 모델을 그대로 씁니다
    tuned_model = trainer.model
    tuned_model.gradient_checkpointing_disable()
else:
    # 세션을 다시 시작했으면 저장된 어댑터를 불러옵니다
    from peft import PeftModel
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )
    tuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

tuned_model.eval()                        # 드롭아웃을 끕니다
tuned_model.config.use_cache = True       # KV 캐시를 켜야 생성이 빠릅니다
tokenizer.truncation_side = "left"        # 질문이 길면 뒤가 아니라 앞을 자릅니다

In [23]:
def answer_for(item, max_new_tokens=200):
    prompt = messages_for(item)["prompt"]
    text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
    with torch.no_grad():
        output = tuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [24]:
print(val_set[0]["question"])

How can I create a list in Python that stores the first 10 numbers divisible by 4?


In [25]:
print(answer_for(val_set[0]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


To achieve this, you can use the following code:

```python
numbers = [i for i in range(10) if i % 4 == 0]
```

In this code, we utilize a list comprehension to generate the desired list. The `range(10)` function generates numbers from 0 to 9 (inclusive). We then apply an `if` condition using the modulo operator `%`, which checks whether each number is divisible by 4. If it is, the number is included in the resulting list.

After executing this code, the variable `numbers` will store the list `[0, 4, 8]`, representing the first three numbers divisible by 4 within the given range.


In [26]:
print(val_set[0]["answer"])

You can use list comprehension in Python to create such a list. Here's an example code snippet that accomplishes this:

```python
divisible_by_four = [i for i in range(1,11) if i % 4 == 0]
```

In this code, we use the `range(1, 11)` function to generate numbers from 1 to 10 (inclusive). Then, we use the `%` operator to check if each number is divisible by 4. If the remainder of the division is 0, the number is considered divisible by 4, and it is included in the resulting list `divisible_by_four`.


In [27]:
with tuned_model.disable_adapter():
    print(answer_for(val_set[0]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


```python
numbers = [i for i in range(4, 41, 4)]
print(numbers)
```
This code creates a list of numbers from 4 to 40 (inclusive) that are divisible by 4 using a list comprehension and prints it.
